# 과제 2 — 학습된 모델로 감성 예측
최고 성능 모델 **Exp3-v8 (TF-IDF + Sweep + 추가 개선)** 을 사용하여 사용자 입력 문장의 감성을 예측

> Dev 70.05% | Test **69.13%** — 베이스라인(67.38%) 대비 **+1.75%p** (최종 제출 모델)

In [1]:
# Cell 0: 패키지 설치
# datasets: Hugging Face 데이터셋 로드 (TF-IDF refit용)
# scikit-learn: TF-IDF 벡터라이저
# torch: MLP 모델 추론
!pip install datasets scikit-learn torch -q

In [2]:
# Cell 1: 패키지 임포트
import re
import numpy as np
import torch
import torch.nn as nn
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer

# GPU 사용 가능 시 GPU, 아니면 CPU 사용
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'사용 디바이스: {device}')

사용 디바이스: cuda


In [3]:
# Cell 2: MLP 클래스 정의
# 학습 시 사용한 동일한 구조의 신경망 모델
class MLP(nn.Module):
    """Multi-Layer Perceptron 분류 모델

    Args:
        input_size: 입력 벡터 차원 (TF-IDF max_features에 맞춰 설정)
        hidden_size: 첫 번째 은닉층 크기 (W&B Sweep 최적값)
        output_size: 출력 클래스 수 (3: Negative/Neutral/Positive)
        dropout_rate: 과적합 방지를 위한 Dropout 비율 (W&B Sweep 최적값)
    """
    def __init__(self, input_size, hidden_size, output_size, dropout_rate=0.0):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size // 2)
        self.fc3 = nn.Linear(hidden_size // 2, output_size)
        self.activation = nn.GELU()
        self.output_act = nn.Softmax(dim=1)
        self.dropout = nn.Dropout(p=dropout_rate)

    def forward(self, x):
        x = self.dropout(self.activation(self.fc1(x)))
        x = self.dropout(self.activation(self.fc2(x)))
        return self.output_act(self.fc3(x))

print('✅ MLP 클래스 정의 완료')

✅ MLP 클래스 정의 완료


In [7]:
# Cell 3: 전처리 함수 / 핸드크래프트 피처 / TF-IDF 정의 및 재학습
# 학습 노트북(Exp3_v8_Final_Sweep.ipynb)과 완전히 동일한 파라미터 사용

from datasets import load_dataset

# ── 전처리 함수 (학습 시와 동일) ──
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = text.replace('`', "'")
    text = text.replace('****', ' bad ')
    text = text.replace('***', ' bad ')
    text = re.sub(r'!{3,}', ' verymuch ! ', text)
    text = re.sub(r'(.)\1{3,}', r'\1\1', text)
    text = re.sub(r"won't", 'will not', text)
    text = re.sub(r"can't", 'cannot', text)
    text = re.sub(r"n't", ' not', text)
    text = re.sub(r"'re", ' are', text)
    text = re.sub(r"'ve", ' have', text)
    text = re.sub(r"'ll", ' will', text)
    text = re.sub(r"'d", ' would', text)
    text = re.sub(r"'m", ' am', text)
    slangs = [
        (r'\bidk\b','i do not know'), (r'\bur\b','your'),
        (r'\bnaw\b','no'),            (r'\bgonna\b','going to'),
        (r'\bwanna\b','want to'),     (r'\blol\b','laughing'),
        (r'\bomg\b','oh my god'),     (r'\bwtf\b','what the'),
        (r'\bugh\b','disgusting'),    (r'\btho\b','though'),
        (r'\bkinda\b','kind of'),     (r'\bcuz\b','because'),
        (r'\bsoo+\b','so'),           (r'\bthx\b','thanks'),
        (r'\byep\b','yes'),           (r'\byup\b','yes'),
        (r'\bnope\b','no'),           (r'\btbh\b','to be honest'),
        (r'\bimo\b','in my opinion'),
    ]
    for pat, rep in slangs:
        text = re.sub(pat, rep, text)
    return text

# ── 핸드크래프트 피처 추출 함수 (6차원, 학습 시와 동일) ──
def extract_handcraft(texts):
    features = []
    for text in texts:
        t = str(text)
        tl = t.lower()
        words = t.split()
        features.append([
            min(t.count('!'), 5),
            min(t.count('?'), 5),
            sum(1 for w in words if w.isupper() and len(w) > 1),
            min(len(words), 50),
            int(bool(re.search(r'http\S+', tl))),
            int(any(e in tl for e in [':)', ':(', ':d', ':/', 'haha', 'hehe', 'lmao'])),
        ])
    return np.array(features, dtype=np.float32)

# ── TF-IDF 재학습 (학습 데이터 로드 후 fit) ──
print('학습 데이터 로드 중...')
ds = load_dataset('Sp1786/multiclass-sentiment-analysis-dataset')
train_texts = ds['train']['text']
print(f'   학습 샘플 수: {len(train_texts)}')

vectorizer = TfidfVectorizer(max_features=30000, preprocessor=preprocess_text, min_df=2)
vectorizer.fit(train_texts)

# input_size = 실제 TF-IDF vocab 크기 + 핸드크래프트 6 (자동 계산)
tfidf_dim = len(vectorizer.vocabulary_)
input_size = tfidf_dim + 6
print('✅ TF-IDF 재학습 완료')
print(f'   TF-IDF 어휘 크기: {tfidf_dim}')
print(f'   최종 입력 차원: {input_size} (TF-IDF {tfidf_dim} + 핸드크래프트 6)')

학습 데이터 로드 중...
   학습 샘플 수: 31232
✅ TF-IDF 재학습 완료
   TF-IDF 어휘 크기: 11657
   최종 입력 차원: 11663 (TF-IDF 11657 + 핸드크래프트 6)


In [8]:
import os
from google.colab import files

# ── Colab에 .pt 파일 업로드 ──
# 실행 후 나타나는 [파일 선택] 버튼을 클릭하여
print('▶ best_model_exp3_v8_final.pt 파일을 선택해 업로드하세요')
uploaded = files.upload()
print('✅ 업로드 완료:', list(uploaded.keys()))

▶ best_model_exp3_v8_final.pt 파일을 선택해 업로드하세요


Saving best_model_exp3_v8_final.pt to best_model_exp3_v8_final (1).pt
✅ 업로드 완료: ['best_model_exp3_v8_final (1).pt']


In [9]:
# Cell 4: 학습된 MLP 체크포인트 로드
# 모델 초기화 (학습 시와 동일한 하이퍼파라미터)
INPUT_SIZE  = input_size  # 30006 (TF-IDF 30000 + 핸드크래프트 6)
HIDDEN_SIZE = 1000        # W&B Sweep 최적값
OUTPUT_SIZE = 3           # 클래스 수 (Negative/Neutral/Positive)
DROPOUT     = 0.4         # W&B Sweep 최적값

model = MLP(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE, DROPOUT).to(device)

# 학습된 가중치 로드 (위 셀에서 업로드한 파일이 /content/ 에 저장됨)
checkpoint_path = 'best_model_exp3_v8_final.pt'
model.load_state_dict(torch.load(checkpoint_path, map_location=device))

# 평가 모드로 전환 (Dropout 비활성화)
model.eval()

print('✅ 학습된 모델 로드 완료')
print(f'   체크포인트: {checkpoint_path}')
print(f'   하이퍼파라미터: input={INPUT_SIZE}, hidden={HIDDEN_SIZE}, dropout={DROPOUT}')
print(f'   성능: Dev 70.05% | Test 69.13% (+1.75%p vs Baseline)')

✅ 학습된 모델 로드 완료
   체크포인트: best_model_exp3_v8_final.pt
   하이퍼파라미터: input=11663, hidden=1000, dropout=0.4
   성능: Dev 70.05% | Test 69.13% (+1.75%p vs Baseline)


In [10]:
# Cell 5: 레이블 매핑 정의
# 모델 출력 (0, 1, 2) → 실제 감성 레이블
# ⚠️ 데이터셋 확인 후 실제 매핑에 맞게 수정 필요
label_map = {
    0: "negative",  # 부정
    1: "neutral",   # 중립
    2: "positive"   # 긍정
}

print('✅ 레이블 매핑:')
for idx, label in label_map.items():
    print(f'   {idx} → {label}')

✅ 레이블 매핑:
   0 → Negative
   1 → Neutral
   2 → Positive


In [11]:
# Cell 6: 예측 함수 정의
def predict_sentiment(sentence):
    """입력 문장의 감성을 예측하는 함수

    처리 과정:
    1. 문장을 TF-IDF 벡터라이저로 변환 (text → sparse 벡터)
    2. dense NumPy 배열 → PyTorch Tensor 변환
    3. MLP 모델에 입력하여 예측
    4. Softmax 출력에서 가장 높은 확률의 클래스 선택
    5. 결과 출력

    Args:
        sentence (str): 예측할 문장

    Returns:
        None (결과를 직접 출력)
    """
    # Step 1: TF-IDF + 핸드크래프트 피처 결합 (학습 시와 동일한 build_features 로직)
    tfidf_mat = vectorizer.transform([sentence])           # sparse (1, 11657) — TF-IDF 피처
    hc_mat    = csr_matrix(extract_handcraft([sentence]))  # sparse (1, 6)     — 핸드크래프트 피처
    combined  = hstack([tfidf_mat, hc_mat]).toarray()      # dense  (1, 11663) — 최종 입력 벡터

    # Step 2: NumPy 배열 → PyTorch Tensor 변환
    tensor = torch.FloatTensor(combined).to(device)

    # Step 3: 모델 예측 (torch.no_grad(): 기울기 계산 비활성화)
    with torch.no_grad():
        output = model(tensor)
        pred_class = torch.argmax(output, dim=1).item()

    # Step 4: 예측 결과 출력
    sentiment = label_map[pred_class]
    print(f"Input sentence: {sentence}")
    print(f"This sentence is {sentiment} sentence.")

print('✅ 예측 함수 정의 완료')

✅ 예측 함수 정의 완료


In [12]:
# Cell 7: 예측 테스트
# 다양한 감성의 문장으로 모델 성능 확인

print('='*60)
print('📝 감성 예측 테스트')
print('='*60)
print()

# 긍정 문장
predict_sentiment("I love this item")
print()

# 부정 문장
predict_sentiment("This is the worst product ever")
print()

# 중립 문장
predict_sentiment("It was okay, nothing special")
print()

print('='*60)

📝 감성 예측 테스트

Input sentence: I love this item
This sentence is Positive sentence.

Input sentence: This is the worst product ever
This sentence is Negative sentence.

Input sentence: It was okay, nothing special
This sentence is Neutral sentence.



In [13]:
# Cell 8: 사용자 입력 (Interactive)
# 직접 문장을 입력하여 예측 결과 확인

print('💬 사용자 입력 모드')
print('   입력 예시: "The movie was absolutely amazing!"')
print('   종료: "quit" 입력')
print()

while True:
    user_input = input('문장 입력: ').strip()

    if user_input.lower() == 'quit':
        print('종료합니다.')
        break

    if not user_input:
        print('⚠️ 문장을 입력해주세요.\n')
        continue

    predict_sentiment(user_input)
    print()

💬 사용자 입력 모드
   입력 예시: "The movie was absolutely amazing!"
   종료: "quit" 입력

문장 입력: I love deep learning
Input sentence: I love deep learning
This sentence is Positive sentence.

문장 입력: I need to sleep
Input sentence: I need to sleep
This sentence is Neutral sentence.

문장 입력: My mom will be angry
Input sentence: My mom will be angry
This sentence is Negative sentence.

문장 입력: quit
종료합니다.
